# BCS 3101: Basics of Machine Learning
## Assignment 2 – Notebook 1

**Student Name:** AYEN GEOFFREY ALEXANDER  
**Registration Number:** 2024/A/KCS/5102/G/F  
**Group Project:** Predicting Monthly Maize and Beans Prices in Selected Ugandan Markets  

This notebook covers **Stages 1 to 5** of the Data Pre-processing & Exploratory Data Analysis Pipeline as given in the official companion guide.

this is what we are going to:  
1. Identify the Machine Learning problem and title.  
2. Source and describe the dataset.  
3. Set up the working environment  
4. Import the required libraries.  
5. Load the data and take a first look.  

---
## Stage 1: Identify Machine Learning Problem and Title
We are not just looking at numbers. We want to answer a real question that someone in Uganda might care about.

**Our problem title:**  
**Predicting Monthly Maize and Bean Prices in Selected Ugandan Markets Using Historical Price and Market Data**



---
## Stage 2: Source and describe Dataset

### Where the data comes from
We use the **Uganda Real-Time Food Prices (RTFP)** dataset.  
It is published by the World Bank Development Economics Data Group.  
The prices are collected from the World Food Programme (WFP) and the Food and Agriculture Organization (FAO).  
Some missing prices are filled by machine-learning estimates, which is why we also have the completed columns that start with `c_`.

Source link (public):  
https://microdata.worldbank.org/catalog/8241  
Also available on the Humanitarian Data Exchange (HDX).

### Why this dataset is suitable
- It is real-world data, not a toy dataset made only for teaching.
- It has more than 500 rows and more than 5 columns.
- It contains genuine data-quality issues (many missing values in the original price columns).
- It relates directly to the machine-learning problem we chose.

### Markets we selected
To keep the work focused and high quality, we keep only seven markets:
1. Market Average (national picture)
2. Gulu
3. Lira
4. Jinja
5. Hoima
6. Busia
7. Mbarara

These markets give good geographic coverage (north, east, west, and national average) and have complete monthly records from 2007 to 2026.

### Target variables
- `c_maize` – completed monthly maize price (UGX per kg)
- `c_beans` – completed monthly beans price (UGX per kg)

We also keep the original observed columns (`maize` and `beans`) so that later notebooks can show proper missing-value analysis.

---
## Stage 3: Set Up working Environment

We can use either Google Colab or a local Jupyter Notebook.  
Both produce the same `.ipynb` file type.  

**Notebook file name (important for marks):**  
`2024AKCS5102GF_BCS3101_Assignment2_Notebook1.ipynb`

---
## Stage 4: Import the Required Libraries

We import every library we will need in the first code cell.  
This is good practice and lets anyone who opens the notebook see the tools at a glance.


In [2]:
# Import the libraries we need for this notebook
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# These will be used more in later notebooks, but we import them now so the whole pipeline is ready
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

# Optional but useful for missing-data pictures later
!pip install missingno
from scipy import stats

# Make plots look clean
%matplotlib inline
sns.set_style('whitegrid')

print("All libraries imported successfully.")

All libraries imported successfully.


**What each library does**  
- `pandas` – loads and works with tables of data  
- `numpy` – fast number calculations  
- `matplotlib` and `seaborn` – draw charts  
- `scikit-learn` – tools for scaling, encoding, splitting, and later modelling  
- `missingno` – special pictures that show where data is missing  
- `scipy.stats` – extra statistical tests if we need them

---
## Stage 5: Load the Data 

Before we clean anything, we must first understand what we have.  
The guide says, "This inspection stage is where you form your hypotheses about the data’s quality; you are not yet fixing anything, you are diagnosing.”

In [4]:
# Load the original Excel file
# Make sure the file UGA_RTFP_mkt_2007_2026-08-24.xlsx is in the same folder as this notebook
# or update the path below if you put the file somewhere else.
!pip install openpyxl

df_full = pd.read_excel('UGA_RTFP_mkt_2007_2026-08-24.xlsx')

print("Full dataset loaded.")
print(f"Rows: {df_full.shape[0]}, Columns: {df_full.shape[1]}")

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------

The full file has more than 10 000 rows because it contains many markets.  
We now keep only the seven markets we chose.

In [5]:
# Keep only the seven markets we decided on
selected_markets = [
    'Market Average',
    'Gulu',
    'Lira',
    'Jinja',
    'Hoima',
    'Busia',
    'Mbarara'
]

df = df_full[df_full['mkt_name'].isin(selected_markets)].copy()

# Reset the index so it is clean
df = df.reset_index(drop=True)

print("After selecting the seven markets:")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("\nMarkets in our working data:")
print(df['mkt_name'].value_counts())

After selecting the seven markets:
Rows: 1652, Columns: 102

Markets in our working data:
mkt_name
Busia             236
Gulu              236
Hoima             236
Jinja             236
Lira              236
Mbarara           236
Market Average    236
Name: count, dtype: int64


We now have a smaller, focused table.  
Each of the seven markets has 236 monthly records (from 2007 to 2026).  
Total rows = 7 × 236 = 1 652. 

### First look at the data
The guide lists several commands that every student must run and interpret.  
We run them one by one and write what we see.

In [6]:
# Look at the first few rows
print("First 5 rows:")
df.head()

First 5 rows:


,ISO3,country,adm1_name,adm2_name,mkt_name,lat,lon,geo_id,price_date,year,...,l_sorghum_fao,c_sorghum_fao,inflation_sorghum_fao,trust_sorghum_fao,o_food_price_index,h_food_price_index,l_food_price_index,c_food_price_index,inflation_food_price_index,trust_food_price_index
0,UGA,Uganda,Busia,Samia-bugwe,Busia,0.47,34.09,gid_4700000340900000,2007-01-01,2007,...,707.48,731.99,NaN,8.8,0.45,0.46,0.44,0.45,NaN,9.0
1,UGA,Uganda,Busia,Samia-bugwe,Busia,0.47,34.09,gid_4700000340900000,2007-02-01,2007,...,707.48,707.98,NaN,8.8,0.45,0.46,0.43,0.43,NaN,9.0
2,UGA,Uganda,Busia,Samia-bugwe,Busia,0.47,34.09,gid_4700000340900000,2007-03-01,2007,...,684.03,706.05,NaN,8.8,0.43,0.44,0.42,0.44,NaN,9.0
3,UGA,Uganda,Busia,Samia-bugwe,Busia,0.47,34.09,gid_4700000340900000,2007-04-01,2007,...,683.61,702.08,NaN,8.8,0.43,0.45,0.42,0.45,NaN,9.0
4,UGA,Uganda,Busia,Samia-bugwe,Busia,0.47,34.09,gid_4700000340900000,2007-05-01,2007,...,680.76,719.04,NaN,8.8,0.45,0.46,0.44,0.45,NaN,9.0


The first rows show the structure.  
We can already see columns for market name, year, month, latitude, longitude, and many price columns.

In [7]:
# Exact shape
print("Shape of our working dataset:")
print(df.shape)
print(f"\nNumber of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

Shape of our working dataset:
(1652, 102)

Number of rows: 1652
Number of columns: 102


In [8]:
# Column names and data types
print("Column names and data types:")
df.info()

Column names and data types:
<class 'pandas.DataFrame'>
RangeIndex: 1652 entries, 0 to 1651
Columns: 102 entries, ISO3 to trust_food_price_index
dtypes: datetime64[us](3), float64(88), int64(3), str(8)
memory usage: 1.3 MB


`df.info()` tells us two important things:  
1. The data type of every column (int, float, object).  
2. How many non-null values each column has.  

We can already see that the original price columns (`beans`, `maize`, etc.) have many missing values, while the completed columns (`c_beans`, `c_maize`, etc.) are full.

In [9]:
# Summary statistics for numeric columns
print("Summary statistics for numeric columns:")
df.describe()

Summary statistics for numeric columns:


,lat,lon,price_date,year,month,start_dense_data,last_survey_point,data_coverage,data_coverage_recent,index_confidence_score,...,l_sorghum_fao,c_sorghum_fao,inflation_sorghum_fao,trust_sorghum_fao,o_food_price_index,h_food_price_index,l_food_price_index,c_food_price_index,inflation_food_price_index,trust_food_price_index
count,1416.000000,1416.000000,1652,1652.000000,1652.000000,1652,1652,1652.00,1.652000e+03,1.652000e+03,...,1652.000000,1652.000000,1568.000000,1652.000000,1652.000000,1652.000000,1652.000000,1652.000000,1568.000000,1652.000000
mean,1.118333,32.418333,2016-10-15 19:55:55.932203,2016.338983,6.432203,2007-01-01 00:00:00,2026-07-01 00:00:00,18.25,2.808000e+01,9.700000e-01,...,1069.277984,1126.218832,5.587519,9.492494,0.899237,0.929231,0.869619,0.899062,6.410319,9.570036
min,-0.610000,30.650000,2007-01-01 00:00:00,2007.000000,1.000000,2007-01-01 00:00:00,2026-07-01 00:00:00,18.25,2.808000e+01,9.700000e-01,...,591.440000,612.160000,-44.490000,8.800000,0.420000,0.430000,0.410000,0.420000,-31.410000,9.000000
25%,0.420000,31.350000,2011-11-23 12:00:00,2011.000000,3.000000,2007-01-01 00:00:00,2026-07-01 00:00:00,18.25,2.808000e+01,9.700000e-01,...,806.635000,845.760000,-4.920000,8.800000,0.690000,0.720000,0.670000,0.690000,-3.702500,9.000000
50%,0.950000,32.610000,2016-10-16 12:00:00,2016.000000,6.000000,2007-01-01 00:00:00,2026-07-01 00:00:00,18.25,2.808000e+01,9.700000e-01,...,1030.170000,1098.595000,4.115000,9.800000,0.900000,0.930000,0.870000,0.900000,6.230000,9.800000
75%,2.230000,33.200000,2021-09-08 12:00:00,2021.000000,9.000000,2007-01-01 00:00:00,2026-07-01 00:00:00,18.25,2.808000e+01,9.700000e-01,...,1277.720000,1357.250000,13.550000,10.000000,1.070000,1.120000,1.032500,1.072500,15.017500,10.000000
max,2.770000,34.090000,2026-08-01 00:00:00,2026.000000,12.000000,2007-01-01 00:00:00,2026-07-01 00:00:00,18.25,2.808000e+01,9.700000e-01,...,1995.080000,2109.940000,86.090000,10.000000,1.600000,1.680000,1.510000,1.580000,64.470000,10.000000
std,1.151934,1.149847,NaN,5.681330,3.439307,NaN,NaN,0.00,1.066137e-14,1.110559e-16,...,304.647581,325.127510,17.649487,0.546420,0.267247,0.275978,0.257610,0.265571,15.013888,0.446649


`df.describe()` gives us the count, mean, standard deviation, minimum, 25 %, 50 % (median), 75 % and maximum for every numeric column.  
This is the first place we can spot strange values (for example a negative price or an extremely high price that looks like a typing error).

In [10]:
# Exact data types
print("Data types of each column:")
print(df.dtypes)

Data types of each column:
ISO3                              str
country                           str
adm1_name                         str
adm2_name                         str
mkt_name                          str
                               ...   
h_food_price_index            float64
l_food_price_index            float64
c_food_price_index            float64
inflation_food_price_index    float64
trust_food_price_index        float64
Length: 102, dtype: object


In [11]:
# List of all column names
print("All column names:")
print(df.columns.tolist())

All column names:
['ISO3', 'country', 'adm1_name', 'adm2_name', 'mkt_name', 'lat', 'lon', 'geo_id', 'price_date', 'year', 'month', 'currency', 'components', 'start_dense_data', 'last_survey_point', 'data_coverage', 'data_coverage_recent', 'index_confidence_score', 'spatially_interpolated', 'beans', 'cassava_flour', 'cassava_flour_fao', 'maize', 'maize_flour', 'millet', 'millet_fao', 'oil', 'salt', 'sorghum', 'sorghum_fao', 'o_beans', 'h_beans', 'l_beans', 'c_beans', 'inflation_beans', 'trust_beans', 'o_cassava_flour', 'h_cassava_flour', 'l_cassava_flour', 'c_cassava_flour', 'inflation_cassava_flour', 'trust_cassava_flour', 'o_cassava_flour_fao', 'h_cassava_flour_fao', 'l_cassava_flour_fao', 'c_cassava_flour_fao', 'inflation_cassava_flour_fao', 'trust_cassava_flour_fao', 'o_maize', 'h_maize', 'l_maize', 'c_maize', 'inflation_maize', 'trust_maize', 'o_maize_flour', 'h_maize_flour', 'l_maize_flour', 'c_maize_flour', 'inflation_maize_flour', 'trust_maize_flour', 'o_millet', 'h_millet', 'l_

Looking at the column names helps us spot any strange characters or inconsistent naming that could cause problems later.

In [12]:
# Count of missing values in every column
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100

missing_df = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_percent': missing_pct.round(2)
}).sort_values('missing_percent', ascending=False)

print("Missing values (highest first):")
print(missing_df[missing_df['missing_count'] > 0])

Missing values (highest first):
                             missing_count  missing_percent
millet_fao                            1652           100.00
sorghum_fao                           1652           100.00
salt                                  1588            96.13
cassava_flour_fao                     1556            94.19
oil                                   1069            64.71
millet                                1032            62.47
sorghum                                974            58.96
cassava_flour                          963            58.29
maize                                  900            54.48
beans                                  871            52.72
maize_flour                            823            49.82
lat                                    236            14.29
lon                                    236            14.29
inflation_beans                         84             5.08
inflation_millet_fao                    84             5.08
inflatio

This table is very important.  
It shows exactly how many values are missing and the percentage.  
Later notebooks (especially Notebook 2 on Data Cleaning) will use these numbers to decide whether to fill the missing values or drop the column.

In [13]:
# Check for exact duplicate rows
dup_count = df.duplicated().sum()
print(f"Number of exact duplicate rows: {dup_count}")

Number of exact duplicate rows: 0


We have 0 duplicate data
Duplicate rows can make some patterns look stronger than they really are.  
If the count was greater than zero we would have removed them in the cleaning notebook.

In [14]:
# Quick look at our two main target columns
print("Summary of the two main target variables:")
print(df[['c_maize', 'c_beans']].describe())

Summary of the two main target variables:
           c_maize      c_beans
count  1652.000000  1652.000000
mean   1078.339189  2667.873311
std     458.971158  1074.227114
min     407.410000   902.660000
25%     729.530000  1855.692500
50%     980.660000  2480.045000
75%    1300.000000  3350.000000
max    2870.680000  6557.420000


Both target columns (`c_maize` and `c_beans`) have no missing values.  
Their means, medians and ranges look reasonable for Ugandan market prices in UGX.

In [15]:
# Save the filtered dataset so the other group members can use the same clean starting point
df.to_csv('uganda_maize_beans_selected_markets.csv', index=False)
print("Filtered dataset saved as 'uganda_maize_beans_selected_markets.csv'")
print("All later notebooks should load this file instead of the original Excel.")

Filtered dataset saved as 'uganda_maize_beans_selected_markets.csv'
All later notebooks should load this file instead of the original Excel.




### What we have done in this notebook
- Chose a clear regression problem and wrote a strong title.
- Described the real-world source of the data and why it meets the assignment rules.
- Selected seven markets that give good coverage and enough rows.
- Set up the environment and imported the libraries listed in the guide.
- Loaded the data, filtered it, and ran every first-look command the guide requires.
- Saved a clean starting CSV for the rest of the group.

### What comes next
Notebook 2 (KUKUNDAKWE SAVIOUS) will take the CSV we just saved and perform full **Data Cleaning** (missing values, duplicates and outliers) following Stage 6 of the companion guide.